# Multitasking experiment: data analysis

<a href="https://colab.research.google.com/github/merettearula1/nai_group/blob/main/analysis/multitask_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook reads every participant CSV that the experiment saved to Google Drive (`multitask_data/`), checks the data, builds one row per participant and makes a first comparison of **single-task vs multitask** performance.

Every participant does both conditions (within-subjects design):

| | Single | Multi |
|---|---|---|
| Video | video alone → 6 questions | video while doing math → 6 questions |
| Math | 5 min math alone | math while the video plays |

**How to use:** Runtime → Run all. Colab asks for permission to access Google Drive. Sign in with the **personal Google account that runs the Apps Script** (the Drive copies are saved there).

The column and event definitions are in the repo's `README.md` → *Data dictionary*.

## 1. Setup

In [ ]:
import glob, json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Colours: single = blue, multi = orange (used in every chart)
COL = {"single": "#2a78d6", "multi": "#eb6834"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

### Where the data is

- `DATA_DIR`: the Drive folder written by the Apps Script.
- `INCLUDE_TEST`: `True` also reads `multitask_data/test/` (runs from `?test=1` / `?fast=1` links). Keep it `False` for the real analysis. Test runs are useful to try the notebook before real data exists.
- Not using Drive? Set `USE_DRIVE = False` and the next cell asks you to upload CSV files instead.

In [ ]:
USE_DRIVE = True
DATA_DIR = "/content/drive/MyDrive/multitask_data"
INCLUDE_TEST = False
OUT_DIR = DATA_DIR + "/analysis"      # where the tidy tables are saved at the end

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif IN_COLAB:
    from google.colab import files
    DATA_DIR = OUT_DIR = "/content/uploaded"
    os.makedirs(DATA_DIR, exist_ok=True)
    for name, data in files.upload().items():
        with open(os.path.join(DATA_DIR, name), "wb") as f:
            f.write(data)

paths = sorted(glob.glob(os.path.join(DATA_DIR, "multitask_*.csv")))
if INCLUDE_TEST:
    paths += sorted(glob.glob(os.path.join(DATA_DIR, "test", "multitask_*.csv")))
print(f"{len(paths)} CSV files found in {DATA_DIR}" + (" (+ test/)" if INCLUDE_TEST else ""))
assert paths, "No files found: check DATA_DIR, or set INCLUDE_TEST = True to try the notebook on test runs."

## 2. Read all files

Each file is one session, one row per event. Text columns are kept as text (e.g. `item_id`, `response`), the numeric ones are converted. `details` is a JSON string; it is parsed into `details_obj`.

In [ ]:
NUM_COLS = ["condition", "event_index", "timestamp_ms", "t_ms", "phase", "item_index",
            "response_index", "correct", "rt_ms"]

def read_one(path):
    d = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    d["source_file"] = os.path.basename(path)
    d["is_test"] = os.path.basename(os.path.dirname(path)) == "test"
    return d

raw = pd.concat([read_one(p) for p in paths], ignore_index=True)
for c in NUM_COLS:
    raw[c] = pd.to_numeric(raw[c].replace("", np.nan), errors="coerce")
raw["details_obj"] = raw["details"].map(lambda s: json.loads(s) if s else {})
print(raw.shape)
raw.head()

### File inventory

One line per file. `complete` = the session reached the end screen (has `experiment_end` and summary rows). Sessions that a participant abandoned only exist if the CSV was saved, so most files should be complete.

In [ ]:
def file_info(d):
    ev = d["event_type"]
    init = d.loc[ev == "session_init", "details_obj"]
    init = init.iloc[0] if len(init) else {}
    return pd.Series({
        "participant_id": d["participant_id"].iloc[0],
        "condition": d["condition"].iloc[0],
        "math_single_list": d["math_single_list"].iloc[0],
        "language": d["language"].iloc[0],
        "started": pd.to_datetime(d["timestamp_ms"].min(), unit="ms", utc=True),
        "duration_min": (d["t_ms"].max() - d["t_ms"].min()) / 60000,
        "n_events": len(d),
        "complete": (ev == "experiment_end").any() and (ev == "summary").any(),
        "fast_mode": bool(init.get("fast_mode", False)),
        "condition_source": init.get("condition_source", ""),
        "math_source": init.get("math_source", ""),
        "is_test": d["is_test"].iloc[0],
    })

files_df = raw.groupby("source_file").apply(file_info).reset_index()
files_df.sort_values("started")

### Duplicates and exclusions

The same participant can have more than one file, e.g. when **Send again** was pressed after an upload error, or a participant ID was reused. We keep one file per participant: the **complete** one with the **most events** (latest if tied). Incomplete sessions and fast-mode runs are dropped. Add IDs to `EXCLUDE_IDS` to remove participants by hand (e.g. pilots); write down why.

In [ ]:
EXCLUDE_IDS = []          # e.g. ["P00", "PILOT1"]

dups = files_df[files_df.duplicated("participant_id", keep=False)]
if len(dups):
    print("Participants with more than one file:")
    display(dups.sort_values(["participant_id", "started"]))

keep = (files_df[files_df["complete"] & ~files_df["fast_mode"] & ~files_df["participant_id"].isin(EXCLUDE_IDS)]
        .sort_values(["participant_id", "n_events", "started"])
        .groupby("participant_id").tail(1))
dropped = files_df[~files_df["source_file"].isin(keep["source_file"])]
print(f"Kept {len(keep)} sessions, dropped {len(dropped)} files")
if len(dropped):
    display(dropped[["source_file", "participant_id", "complete", "fast_mode", "n_events"]])

ev = raw[raw["source_file"].isin(keep["source_file"])].copy()

## 3. One row per participant

Built from the `summary` rows (written at the end of each session) plus the background questions. Derived measures:

- **math rate** = correct answers per minute. Math alone always lasts 5 min; during multitasking it lasts as long as the video (`math_multi_duration_sec`), so the counts are not directly comparable but the rates are.
- **math accuracy** = correct / attempted.
- **video score** = correct out of 6 (timed-out questions count as wrong).

In [ ]:
summ = (ev[ev["event_type"] == "summary"]
        .pivot_table(index="participant_id", columns="item_id", values="response", aggfunc="first"))
TEXT_SUMMARY = ["video_single_id", "video_multi_id", "math_single_list", "math_multi_list", "block_order"]
for c in summ.columns.difference(TEXT_SUMMARY):
    summ[c] = pd.to_numeric(summ[c], errors="coerce")

demo = (ev[ev["event_type"] == "demographic"]
        .pivot_table(index="participant_id", columns="item_id", values="response", aggfunc="first"))
if "age" in demo:
    demo["age"] = pd.to_numeric(demo["age"], errors="coerce")

meta = keep.set_index("participant_id")[["condition", "language", "started", "duration_min", "source_file"]]
wide = meta.join(demo).join(summ.drop(columns=["math_single_list"], errors="ignore"))
wide = wide.join(keep.set_index("participant_id")["math_single_list"])
# Files from before the order change have no block_order row: they all ran single tasks first.
wide["block_order"] = wide["block_order"].fillna("single_first") if "block_order" in wide else "single_first"

wide["math_single_rate"] = wide["math_single_correct"] / 5
wide["math_multi_rate"] = wide["math_multi_correct"] / (wide["math_multi_duration_sec"] / 60)
wide["math_single_acc"] = wide["math_single_correct"] / wide["math_single_attempted"]
wide["math_multi_acc"] = wide["math_multi_correct"] / wide["math_multi_attempted"]
wide["math_single_mean_rt_s"] = wide["math_single_mean_rt_ms"] / 1000
wide["math_multi_mean_rt_s"] = wide["math_multi_mean_rt_ms"] / 1000
wide

## 4. Sample description

In [ ]:
print("N =", len(wide))
display(wide[["age"]].describe().T if "age" in wide else "no age column")
for c in ["gender", "education", "language"]:
    if c in wide:
        display(wide[c].value_counts().rename_axis(c).to_frame("n"))

print("Counterbalancing (should be roughly equal):")
pd.crosstab(wide["condition"], wide["math_single_list"], margins=True)

## 5. Data quality

Per participant: how often they left the tab or fullscreen, total time away from the tab, whether the videos ended normally, and how the ERR video was started. Look for outliers before running the tests. Decide exclusion rules **before** looking at the results, and add the IDs to `EXCLUDE_IDS` above.

In [ ]:
def time_hidden_s(d):
    # total seconds between tab_hidden and the following tab_visible
    d = d.sort_values("t_ms")
    total, start = 0.0, None
    for etype, t in zip(d["event_type"], d["t_ms"]):
        if etype == "tab_hidden":
            start = t
        elif etype == "tab_visible" and start is not None:
            total += t - start; start = None
    return total / 1000

def quality(d):
    e = d["event_type"]
    ends = d.loc[e == "video_end", "details_obj"].map(lambda x: x.get("reason", ""))
    plays = d.loc[(e == "video_state") & (d["response"] == "play_assumed"), "details_obj"]
    return pd.Series({
        "tab_hidden_n": (e == "tab_hidden").sum(),
        "tab_hidden_s": time_hidden_s(d),
        "fullscreen_exit_n": (e == "fullscreen_exit").sum(),
        "unload_attempt_n": (e == "unload_attempt").sum(),
        "video_errors_n": e.isin(["video_error", "youtube_api_load_error"]).sum(),
        "video_end_reasons": ", ".join(ends),
        "err_play_trigger": ", ".join(plays.map(lambda x: x.get("trigger", ""))),
        "manual_start_clicks": (e == "video_manual_start_click").sum(),
        "video_single_timed_out": None, "video_multi_timed_out": None,
    })

qc = ev.groupby("participant_id").apply(quality)
qc["video_single_timed_out"] = wide["video_single_timed_out"]
qc["video_multi_timed_out"] = wide["video_multi_timed_out"]
qc.style.background_gradient(subset=["tab_hidden_n", "tab_hidden_s", "fullscreen_exit_n"], cmap="Oranges")

## 6. Paired t-test: single vs multitask

Each participant did both conditions, so we compare each person with themselves: a **paired (dependent-samples) t-test** on the difference *multi − single*.

**Hypotheses** (for each measure)
- H0: the mean difference multi − single is 0.
- H1: the mean difference is not 0 (two-sided). Expected direction: worse when multitasking (lower video score, lower math rate and accuracy, slower math).

**Measures.** Decide *before* looking at the results which one is primary, and write it down. The default below has two primary outcomes (one per task) and two secondary ones. The Holm correction is applied within each group.

| Measure | Single | Multi | Type |
|---|---|---|---|
| Video score (correct of 6) | `video_single_correct` | `video_multi_correct` | primary |
| Math rate (correct / min) | `math_single_rate` | `math_multi_rate` | primary |
| Math accuracy | `math_single_acc` | `math_multi_acc` | secondary |
| Math mean RT (s) | `math_single_mean_rt_s` | `math_multi_mean_rt_s` | secondary |

In [ ]:
MEASURES = {
    # name: (single column, multi column, "primary"/"secondary")
    "Video score (correct of 6)": ("video_single_correct", "video_multi_correct", "primary"),
    "Math rate (correct / min)":  ("math_single_rate", "math_multi_rate", "primary"),
    "Math accuracy":              ("math_single_acc", "math_multi_acc", "secondary"),
    "Math mean RT (s)":           ("math_single_mean_rt_s", "math_multi_mean_rt_s", "secondary"),
}
ALPHA = 0.05

def pair(name):
    s, m, _ = MEASURES[name]
    d = wide[[s, m]].astype(float).dropna()
    return d[s], d[m]

# The paired data: one row per participant, with the difference
paired_data = pd.concat({name: pd.DataFrame({"single": pair(name)[0], "multi": pair(name)[1]})
                         .assign(diff=lambda x: x["multi"] - x["single"])
                         for name in MEASURES}, axis=1)
paired_data.round(2)

### 6a. Assumption checks

The paired t-test assumes the **differences** (not the raw scores) are roughly normally distributed and have no extreme outliers.
- **Shapiro–Wilk** on the differences: p < .05 suggests non-normality. With small N it has little power, so look at the plots too.
- **Outliers:** differences more than 3 SD from the mean difference are flagged. Don't drop them automatically; check the data-quality table (section 5) for a reason first.
- If the assumptions look badly violated, report the **Wilcoxon signed-rank test** (computed below) as the main test instead. The video score has only 7 possible values (0–6), so check it especially carefully.

In [ ]:
checks = {}
for name in MEASURES:
    a, b = pair(name)
    diff = b - a
    z = (diff - diff.mean()) / diff.std() if diff.std() > 0 else diff * 0
    checks[name] = {
        "n": len(diff),
        "shapiro_W": stats.shapiro(diff).statistic if len(diff) >= 3 and diff.std() > 0 else np.nan,
        "shapiro_p": stats.shapiro(diff).pvalue if len(diff) >= 3 and diff.std() > 0 else np.nan,
        "skew": stats.skew(diff) if len(diff) >= 3 else np.nan,
        "outliers_|z|>3": ", ".join(diff.index[z.abs() > 3]) or "-",
    }
display(pd.DataFrame(checks).T.round(3))

fig, axes = plt.subplots(2, len(MEASURES), figsize=(3.2 * len(MEASURES), 5.6))
for j, name in enumerate(MEASURES):
    a, b = pair(name)
    diff = b - a
    axes[0, j].hist(diff, bins="auto", color=COL["multi"], edgecolor="white", linewidth=1.5)
    axes[0, j].axvline(0, color="0.3", lw=1, ls="--")
    axes[0, j].set_title(name, fontsize=10); axes[0, j].set_xlabel("multi − single")
    stats.probplot(diff, dist="norm", plot=axes[1, j])
    axes[1, j].get_lines()[0].set(color=COL["multi"], markersize=4)
    axes[1, j].get_lines()[1].set(color="0.3", lw=1)
    axes[1, j].set_title("Q–Q plot", fontsize=10)
fig.tight_layout()

### 6b. The tests

For each measure:
- **mean_diff**: mean of multi − single, with its **95% CI**
- **t(df), p**: paired t-test (two-sided)
- **p_holm**: p adjusted with the Holm correction, applied within the primary and within the secondary measures
- **d<sub>z</sub>**: Cohen's d<sub>z</sub> = mean difference / SD of the differences (about 0.2 small, 0.5 medium, 0.8 large)
- **d_av**: mean difference / average SD of the two conditions. Easier to compare with between-subjects studies; report it next to d<sub>z</sub> if you like.
- **Wilcoxon W, p**: non-parametric check. If it agrees with the t-test, the result doesn't depend on the normality assumption.

In [ ]:
from statsmodels.stats.multitest import multipletests

def paired_test(a, b):
    diff = b - a
    n = len(diff)
    out = {"n": n, "single_M": a.mean(), "single_SD": a.std(), "multi_M": b.mean(), "multi_SD": b.std(),
           "mean_diff": diff.mean(), "SD_diff": diff.std()}
    if n >= 2 and diff.std() > 0:
        t = stats.ttest_rel(b, a)
        ci = t.confidence_interval(1 - ALPHA)
        out.update(df=n - 1, t=t.statistic, p=t.pvalue, CI_low=ci.low, CI_high=ci.high,
                   dz=diff.mean() / diff.std(),
                   d_av=diff.mean() / ((a.std() + b.std()) / 2))
    if n >= 2 and (diff != 0).any():
        w = stats.wilcoxon(b, a)
        out.update(wilcoxon_W=w.statistic, wilcoxon_p=w.pvalue)
    return out

results = pd.DataFrame({name: paired_test(*pair(name)) for name in MEASURES}).T
results["type"] = [MEASURES[n][2] for n in results.index]
results["p_holm"] = np.nan
for typ, idx in results.groupby("type").groups.items():
    p = results.loc[idx, "p"].astype(float)
    ok = p.notna()
    if ok.any():
        results.loc[p[ok].index, "p_holm"] = multipletests(p[ok], alpha=ALPHA, method="holm")[1]
results["significant"] = results["p_holm"] < ALPHA

cols = ["type", "n", "single_M", "single_SD", "multi_M", "multi_SD", "mean_diff", "CI_low", "CI_high",
        "df", "t", "p", "p_holm", "significant", "dz", "d_av", "wilcoxon_W", "wilcoxon_p"]
results = results.reindex(columns=cols)
results.style.format(precision=3, na_rep="–").format({"n": "{:.0f}", "df": "{:.0f}"}, na_rep="–")

### 6c. Reporting the results (APA)

A complete statistical statement for a paired *t*-test includes:

1. **Descriptive statistics** for both conditions (*M* and *SD*).
2. **The test statistic** *t*, its degrees of freedom (*df* = *N* − 1) and the exact *p*-value. Write *p* < .001 for very small values, and drop the leading zero (*p* = .032, not 0.032).
3. **The mean difference** (multi − single) and its 95% confidence interval.
4. **The effect size**: Cohen's *d*<sub>z</sub> for paired samples (about 0.2 small, 0.5 medium, 0.8 large).
5. *(Recommended)* **The non-parametric equivalent**, the Wilcoxon signed-rank test, especially when the differences are skewed or have outliers (section 6a).

**General template**

> Participants scored significantly [higher / lower] in the multitask condition (*M* = XX, *SD* = XX) compared to the single-task condition (*M* = XX, *SD* = XX), *t*(*df*) = X.XX, *p* = .XXX, mean difference = X.XX, 95% CI [lower, upper], *d*<sub>z</sub> = X.XX.

If the difference is not significant, write instead: *Scores did not differ significantly between the multitask (M = …) and single-task (M = …) conditions, t(df) = …, p = …*, and still report the CI and effect size.

The cell below fills in the template for each measure. Significance is decided with the **Holm-adjusted** *p* (section 6b), and the uncorrected *p* is reported next to it. The output is shown formatted for reading and also as plain text to copy into Word.

In [ ]:
from IPython.display import Markdown

# How each measure is described: (subject, word for "higher", word for "lower")
PHRASES = {
    "Video score (correct of 6)": ("Participants answered {sig}{dir} comprehension questions correctly", "more", "fewer"),
    "Math rate (correct / min)":  ("Participants solved {sig}{dir} arithmetic problems correctly per minute", "more", "fewer"),
    "Math accuracy":              ("Arithmetic accuracy was {sig}{dir}", "higher", "lower"),
    "Math mean RT (s)":           ("Participants were {sig}{dir} at solving arithmetic problems", "slower", "faster"),
}

def p_txt(p, it="*"):
    return f"{it}p{it} < .001" if p < .001 else f"{it}p{it} = " + f"{p:.3f}".lstrip("0")

def apa_sentence(name, r, it="*"):
    # it = "*" gives Markdown italics, "" gives plain text
    M, SD, t, dz = (f"{it}{x}{it}" for x in ("M", "SD", "t", "d"))
    dz = dz + ("_z" if not it else "<sub>z</sub>")
    if pd.isna(r["t"]):
        return f"{name}: not enough data or no variation in the differences."
    sig = r["p_holm"] < ALPHA
    stats_txt = (f"{t}({r.df:.0f}) = {r.t:.2f}, {p_txt(r.p, it)} (Holm-adjusted {p_txt(r.p_holm, it)}), "
                 f"mean difference = {r.mean_diff:.2f}, 95% CI [{r.CI_low:.2f}, {r.CI_high:.2f}], {dz} = {r.dz:.2f}")
    multi = f"({M} = {r.multi_M:.2f}, {SD} = {r.multi_SD:.2f})"
    single = f"({M} = {r.single_M:.2f}, {SD} = {r.single_SD:.2f})"
    if sig:
        subj, up, down = PHRASES.get(name, (name + " was {sig}{dir}", "higher", "lower"))
        main = (subj.format(sig="significantly ", dir=up if r.mean_diff > 0 else down)
                + f" in the multitask condition {multi} compared to the single-task condition {single}, {stats_txt}.")
    else:
        main = (f"{name} did not differ significantly between the multitask condition {multi} "
                f"and the single-task condition {single}, {stats_txt}.")
    if pd.notna(r.get("wilcoxon_p")):
        agrees = (r.wilcoxon_p < ALPHA) == (r.p < ALPHA)
        main += (f" A Wilcoxon signed-rank test {'led to the same conclusion' if agrees else 'did not support this result'}, "
                 f"{it}W{it} = {r.wilcoxon_W:.1f}, {p_txt(r.wilcoxon_p, it)}.")
    return main

display(Markdown("\n\n".join(f"**{name}.** {apa_sentence(name, r)}" for name, r in results.iterrows())))

print("\n--- Plain text to copy ---\n")
for name, r in results.iterrows():
    print(apa_sentence(name, r, it="") + "\n")

### 6d. Sample size and power

How many participants a paired t-test needs to detect an effect of a given size d<sub>z</sub> with 80% power (α = .05, two-sided), and what power the current N gives. Useful for planning recruitment. Use the observed d<sub>z</sub> with care: with small N it is a noisy estimate.

In [ ]:
from statsmodels.stats.power import TTestPower
power = TTestPower()

plan = pd.DataFrame({"effect d_z": [0.3, 0.5, 0.8]})
plan["N needed (80% power)"] = [int(np.ceil(power.solve_power(effect_size=d, alpha=ALPHA, power=0.8)))
                                for d in plan["effect d_z"]]
plan["power with current N"] = [power.power(effect_size=d, nobs=len(wide), alpha=ALPHA)
                                if len(wide) >= 2 else np.nan for d in plan["effect d_z"]]
print(f"Current N = {len(wide)}")
plan.round(2)

In [ ]:
fig, axes = plt.subplots(1, len(MEASURES), figsize=(3.2 * len(MEASURES), 3.6))
for ax, (title, (s, m, _)) in zip(axes, MEASURES.items()):
    d = wide[[s, m]].dropna().astype(float)
    for _, r in d.iterrows():                       # one grey line per participant
        ax.plot([0, 1], [r[s], r[m]], color="0.75", lw=1, zorder=1)
    rng = np.random.default_rng(0)
    for x, col, cond in [(0, s, "single"), (1, m, "multi")]:
        ax.scatter(np.full(len(d), x) + rng.uniform(-0.04, 0.04, len(d)), d[col],
                   s=22, color=COL[cond], edgecolor="white", linewidth=0.8, zorder=2)
        ax.errorbar(x + (0.15 if x else -0.15), d[col].mean(), yerr=d[col].sem(),
                    fmt="o", ms=7, color=COL[cond], capsize=3, zorder=3)
    ax.set_xticks([0, 1], ["Single", "Multi"]); ax.set_xlim(-0.4, 1.4)
    ax.set_title(title, fontsize=10); ax.grid(axis="x", visible=False)
fig.suptitle("Single vs multitask (lines = participants, big dot = mean ± SE)", fontsize=11)
fig.tight_layout()

### 6e. Figure 1 for the paper

The same comparison, formatted for the report: y-axis labels with units, no title inside the image (the title and note go in the caption in Word). Whole-number scores (video) get a small vertical offset so overlapping participants stay visible. Saved as a 300 dpi PNG in `OUT_DIR`.

In [ ]:
# --- Figure 1 for the paper ---
import matplotlib.ticker as mticker
PANELS = [  # (single column, multi column, y-axis label, panel title)
    ("video_single_correct", "video_multi_correct", "Correct answers (of 6)", "Video comprehension"),
    ("math_single_rate", "math_multi_rate", "Correct problems per minute", "Arithmetic rate"),
    ("math_single_acc", "math_multi_acc", "Correct answers (%)", "Arithmetic accuracy"),
    ("math_single_mean_rt_s", "math_multi_mean_rt_s", "Mean time per problem (s)", "Arithmetic speed"),
]
# PANELS = PANELS[:2] + PANELS[3:]   # uncomment to leave out the accuracy panel

NROWS = (len(PANELS) + 1) // 2          # 2 panels per row: video + rate on top, accuracy + speed below
fig, axes = plt.subplots(NROWS, 2, figsize=(9, 4 * NROWS))
axes = axes.ravel()
for ax in axes[len(PANELS):]:           # hide an empty slot (e.g. when the accuracy panel is left out)
    ax.set_visible(False)
rng = np.random.default_rng(1)
for i, (ax, (s, m, ylabel, title)) in enumerate(zip(axes, PANELS)):
    d = wide[[s, m]].dropna().astype(float)
    # whole-number scores overlap: add a small vertical offset so every participant is visible
    discrete = (d.round() == d).all().all()
    jy = rng.uniform(-0.12, 0.12, (len(d), 2)) if discrete else np.zeros((len(d), 2))
    jx = rng.uniform(-0.05, 0.05, len(d))
    for k, (_, r) in enumerate(d.iterrows()):
        ax.plot([0 + jx[k], 1 + jx[k]], [r[s] + jy[k, 0], r[m] + jy[k, 1]], color="0.7", lw=0.8, zorder=1)
    for x, col, cond, j in [(0, s, "single", 0), (1, m, "multi", 1)]:
        ax.scatter(x + jx, d[col] + jy[:, j], s=16, color=COL[cond], edgecolor="white", linewidth=0.6, zorder=2)
        ax.errorbar(x + (0.22 if x else -0.22), d[col].mean(), yerr=d[col].sem(), fmt="o", ms=7,
                    color=COL[cond], ecolor="black", elinewidth=1.2, capsize=3, zorder=3)
    ax.set_xticks([0, 1], ["Single-task", "Multitask"]); ax.set_xlim(-0.45, 1.45)
    ax.set_ylabel(ylabel); ax.grid(axis="x", visible=False)
    ax.set_title(title, fontsize=11)
    ax.tick_params(labelsize=10); ax.yaxis.label.set_size(11)
    if "acc" in s:                      # accuracy as percentages, ticks on whole percents
        ax.yaxis.set_major_locator(mticker.MaxNLocator(steps=[1, 2, 5, 10]))
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
fig.tight_layout()
os.makedirs(OUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUT_DIR, "figure1.png"), dpi=300, bbox_inches="tight")
print("Saved", os.path.join(OUT_DIR, "figure1.png"))

## 7. Video questions by item and difficulty

Each video has 6 questions labelled `easy` / `intermediate` / `difficult`. A timed-out question counts as wrong.

In [ ]:
qa = ev[ev["event_type"].isin(["question_answer", "question_timeout"])].copy()
# question_timeout rows have no difficulty: take it from the question_shown row of the same item
diff_map = (ev[ev["event_type"] == "question_shown"]
            .drop_duplicates("item_id").set_index("item_id")["difficulty"])
qa["difficulty"] = qa["item_id"].map(diff_map).fillna(qa["difficulty"])

by_diff = (qa.groupby(["difficulty", "task_condition"])["correct"].mean()
           .unstack().reindex(["easy", "intermediate", "difficult"]))
display(by_diff.round(2))

ax = by_diff[["single", "multi"]].plot.bar(color=[COL["single"], COL["multi"]], width=0.7,
                                            edgecolor="white", linewidth=2, figsize=(5, 3.4), rot=0)
ax.set_ylabel("Proportion correct"); ax.set_ylim(0, 1); ax.set_xlabel("")
ax.set_title("Video questions: accuracy by difficulty"); ax.legend(frameon=False)
ax.grid(axis="x", visible=False)

print("Per item (rows = question ID, e.g. A3 = video A question 3):")
qa.pivot_table(index="item_id", columns="task_condition", values="correct", aggfunc=["mean", "count"]).round(2)

### Response time on video questions

`rt_ms` = time from the question appearing to pressing Continue.

In [ ]:
ans = ev[ev["event_type"] == "question_answer"]
rt_q = ans.groupby(["participant_id", "task_condition"])["rt_ms"].median().unstack() / 1000
display(rt_q.describe().round(1))
d = rt_q[["single", "multi"]].dropna()
pd.Series(paired_test(d["single"], d["multi"]), name="Video question median RT (s)").round(3)

## 8. Math over time

Each math answer with its time since the task started. Shows whether people slowed down or sped up, and whether they kept doing math during the video. Only the first 5 full minutes are shown: math alone stops at 5:00, and during multitasking the last minute is cut short by the end of the video.

In [ ]:
m = ev[ev["event_type"].isin(["math_answer"])].copy()
starts = (ev[ev["event_type"] == "task_start"]
          .set_index(["participant_id", "task_condition"])["t_ms"])
m["t_task_s"] = (m["t_ms"] - starts.reindex(list(zip(m["participant_id"], m["task_condition"]))).values) / 1000
m["minute"] = (m["t_task_s"] // 60).astype("Int64") + 1
m = m[m["minute"] <= 5]

per_min = (m.groupby(["task_condition", "minute"])
             .agg(answers=("correct", "size"), acc=("correct", "mean"), rt_s=("rt_ms", lambda x: x.median() / 1000))
             .reset_index())
per_min["answers_per_person"] = per_min["answers"] / ev["participant_id"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for cond in ["single", "multi"]:
    d = per_min[per_min["task_condition"] == cond]
    axes[0].plot(d["minute"], d["answers_per_person"], marker="o", lw=2, color=COL[cond], label=cond)
    axes[1].plot(d["minute"], d["rt_s"], marker="o", lw=2, color=COL[cond], label=cond)
axes[0].set(title="Math answers per person per minute", xlabel="Minute of task")
axes[1].set(title="Median time per problem (s)", xlabel="Minute of task")
axes[0].legend(frameon=False); fig.tight_layout()

## 9. Counterbalancing checks

Do the results depend on **which video** or **which math list** was used? If one video is much harder, compare conditions within video, or add video as a factor in a model. Conditions 1–2 use videos A/B (Estonian), 3–4 use C/D (English).

In [ ]:
vid = pd.concat([
    wide[["video_single_id", "video_single_correct"]].set_axis(["video", "correct"], axis=1).assign(task="single"),
    wide[["video_multi_id", "video_multi_correct"]].set_axis(["video", "correct"], axis=1).assign(task="multi"),
])
display(vid.pivot_table(index="video", columns="task", values="correct", aggfunc=["mean", "count"]).round(2))

ml = pd.concat([
    wide[["math_single_list", "math_single_rate"]].set_axis(["list", "rate"], axis=1).assign(task="single"),
    wide[["math_multi_list", "math_multi_rate"]].set_axis(["list", "rate"], axis=1).assign(task="multi"),
])
display(ml.pivot_table(index="list", columns="task", values="rate", aggfunc=["mean", "count"]).round(2))

print("Multitask cost by language (multi − single):")
wide.assign(video_cost=wide["video_multi_correct"] - wide["video_single_correct"],
            math_rate_cost=wide["math_multi_rate"] - wide["math_single_rate"]
            ).groupby("language")[["video_cost", "math_rate_cost"]].agg(["mean", "count"]).round(2)

## 10. Optional: mixed model

With enough participants, a mixed model handles the video/list differences at the same time:
`correct ~ task_condition + video + (1 | participant)` on the item-level video answers. A logistic mixed model would be more correct for 0/1 answers; this linear one is a quick first look.

In [ ]:
import statsmodels.formula.api as smf
if qa["participant_id"].nunique() >= 6:
    qa["video"] = qa["item_id"].str[0]
    mm = smf.mixedlm("correct ~ C(task_condition, Treatment('single')) + C(video)",
                     qa.dropna(subset=["correct"]), groups=qa.dropna(subset=["correct"])["participant_id"]).fit()
    print(mm.summary())
else:
    print("Fewer than 6 participants: skipped.")

## 11. Save the tidy tables

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
wide.to_csv(os.path.join(OUT_DIR, "participants_wide.csv"))
ev.drop(columns=["details_obj"]).to_csv(os.path.join(OUT_DIR, "events_all.csv"), index=False)
results.to_csv(os.path.join(OUT_DIR, "paired_results.csv"))
qc.to_csv(os.path.join(OUT_DIR, "data_quality.csv"))
print("Saved to", OUT_DIR)